<a href="https://colab.research.google.com/github/PilliSrilekha1419/ai-mentor-portfolio/blob/main/Day10_Sprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q "numpy<2.0"

!pip install -q --upgrade \
crewai \
litellm \
google-generativeai \
google-genai \
chromadb \
sentence-transformers \
crewai-tools

In [2]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

Enter Gemini API Key: ··········
API key set


In [3]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

10:13:36 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
10:13:36 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [4]:
llm = "gemini/gemini-2.5-flash"

print(llm)

gemini/gemini-2.5-flash


In [5]:
client_db = PersistentClient(path='./chroma_db')

col = client_db.get_or_create_collection('placement_kb')

embed = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Vector DB ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB ready


In [6]:
@crewai_tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base.
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results['documents'][0]

    return '\n---\n'.join(docs)

print("RAG tool created")

RAG tool created


In [7]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

print("Researcher agent ready")

Researcher agent ready


In [8]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Interviewer agent ready")

Interviewer agent ready


In [9]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Coach agent ready")

Coach agent ready


In [10]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Tracker agent ready")

Tracker agent ready


In [11]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

print("Profiles loaded")

Profiles loaded


In [12]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

print("Task builder ready")

Task builder ready


In [13]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

print("Crew created")

Crew created


In [17]:
import asyncio

async def run_crew_safe():

    try:
        result = await crew.kickoff_async()

        print("\n=== FINAL OUTPUT ===\n")
        print(result)

        return result

    except Exception as e:

        print("\n❌ Crew Execution Failed")
        print("Error Type:", type(e).__name__)
        print("Error Message:", str(e)[:300])

        print("\n💡 Fix Suggestions:")
        print("1. Gemini quota exceeded (429 error)")
        print("2. Switch to Ollama or Groq")
        print("3. Wait 30–60 seconds and retry")

        return None


# ✅ RUN IN COLAB
result = await run_crew_safe()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 89990e56-ac6b-48bd-a668-c9ab50daed21                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 49b819f1-c00d-40ea-860a-7ed77d7f160d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#10) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: rag_search                                                                                               │
│  Args: {'query': 'TCS Digital placement preparation details for CSE students'}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: ...

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Create JSON-style progress summary for S001.                                                             │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')


❌ Crew Execution Failed
Error Type: ServerError
Error Message: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

💡 Fix Suggestions:
1. Gemini quota exceeded (429 error)
2. Switch to Ollama or Groq
3. Wait 30–60 seconds and retry


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 89990e56-ac6b-48bd-a668-c9ab50daed21                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
import asyncio

transcripts = []

async def run_safe_crew(p):

    try:
        print("\n" + "="*60)
        print(f"Running for {p['name']} → {p['target_company']}")
        print("="*60)

        crew = Crew(
            agents=[researcher, interviewer, coach, tracker],
            tasks=build_tasks(p),
            process=Process.sequential,
            verbose=True,
        )

        result = await crew.kickoff_async()

        return {
            "student": p["name"],
            "target": p["target_company"],
            "final_output": str(result),
            "status": "success"
        }

    except Exception as e:

        print("\n❌ Crew failed for:", p['name'])
        print("Error:", str(e)[:250])

        # ✅ async-safe delay (IMPORTANT FIX)
        print("⏳ Waiting 30 seconds before next student...\n")
        await asyncio.sleep(30)

        return {
            "student": p["name"],
            "target": p["target_company"],
            "final_output": f"FAILED: {str(e)[:200]}",
            "status": "failed"
        }


# =========================
# MAIN LOOP (SAFE EXECUTION)
# =========================

for p in profiles:

    result = await run_safe_crew(p)
    transcripts.append(result)

print("\nCompleted:", len(transcripts))


Running for Ravi Kumar → TCS Digital


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3e704d11-dca7-4b75-be7a-32ab0947d570                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 3f238c45-70e7-4c83-800e-d8dad75005a5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 14.228699157s.


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Research placement preparation details for Amazon for a IT student.                                      │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


❌ Crew failed for: Arun Pillai
Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your c
⏳ Waiting 30 seconds before next student...



╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: ac268961-60f8-408a-a773-8624e3aec94b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Completed: 3


In [20]:
import json
from pathlib import Path

# convert safely (handles CrewAI objects too)
safe_transcripts = json.loads(json.dumps(transcripts, default=str))

Path("day10_sprint5_transcripts.json").write_text(
    json.dumps(safe_transcripts, indent=2),
    encoding="utf-8"
)

print("Saved transcripts successfully")

Saved transcripts successfully


In [21]:
from pathlib import Path

md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    # safely convert output to string
    md += str(t["final_output"])

    md += "\n\n---\n\n"

# save markdown file
Path("day10_sprint5_report.md").write_text(
    md,
    encoding="utf-8"
)

print("Markdown report created successfully")

Markdown report created successfully


In [22]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>